In [1]:
import numpy as np
import pandas as pd
from scipy.optimize import minimize
import matplotlib.pyplot as plt

In [2]:
mu = np.array([0.130, 0.105, 0.120, 0.110, 0.090, 0.085, 0.071, 0.074])
carbon = np.array([12, 5, 450, 680, 80, 1250, 0, 0])
asset_durations = np.array([6.2, 5.8, 5.5, 5.1, 4.8, 4.2, 7.5, 18.2])
target_liability_duration = 13.5 # Matching 30-year liability modified duration

volatilities = np.array([0.28, 0.22, 0.25, 0.23, 0.16, 0.24, 0.06, 0.09])
corr_matrix = np.eye(8)
for i in range(6):
    for j in range(6):
        if i != j: corr_matrix[i, j] = 0.45
corr_matrix[6, 7] = corr_matrix[7, 6] = 0.85
cov_matrix = np.outer(volatilities, volatilities) * corr_matrix

class PensionALMOptimizer:
    def __init__(self, mu, cov, carbon, durations, target_duration):
        self.mu = mu
        self.cov = cov
        self.carbon = carbon
        self.durations = durations
        self.target_duration = target_duration
        self.n = len(mu)

    def optimize(self, risk_aversion=1.5, esg_penalty=0.0005, max_duration_gap=1.5, min_fixed_income=0.55):
        def objective(w):
            port_var = np.dot(w.T, np.dot(self.cov, w))
            port_ret = np.dot(w, self.mu)
            port_carb = np.dot(w, self.carbon)
            return port_var - (risk_aversion * port_ret) + (esg_penalty * port_carb)

        constraints = [
            {'type': 'eq', 'fun': lambda w: np.sum(w) - 1.0}, # Full allocation
            {'type': 'ineq', 'fun': lambda w: max_duration_gap - abs(np.dot(w, self.durations) - self.target_duration)}, # Duration gap
            {'type': 'ineq', 'fun': lambda w: np.sum(w[-2:]) - min_fixed_income} # Fixed income floor (>= 55%)
        ]

        bounds = tuple((0.0, 0.40) for _ in range(self.n))
        init_w = np.ones(self.n) / self.n

        res = minimize(objective, init_w, method='SLSQP', bounds=bounds, constraints=constraints)
        return res.x

In [3]:
opt = PensionALMOptimizer(mu, cov_matrix, carbon, asset_durations, target_liability_duration)
weights = opt.optimize()

asset_names = ['Adani Green', 'NHPC', 'Tata Power', 'NTPC', 'Power Grid', 'Coal India', '10Y G-Sec', '30Y Green Bond']

df_results = pd.DataFrame({
    'Asset Class': asset_names,
    'Optimal Weight': weights
})

print("--- OPTIMAL STRATEGIC ASSET ALLOCATION (PwC ALM MODEL) ---")
display(df_results.style.format({'Optimal Weight': '{:.2%}'}))

port_ret = np.dot(weights, mu)
port_dur = np.dot(weights, asset_durations)
port_carb = np.dot(weights, carbon)

print(f"\nPortfolio Expected Return: {port_ret:.2%}")
print(f"Portfolio Duration: {port_dur:.2f} Years (Target Liability Duration: {target_liability_duration:.2f} Years)")
print(f"Portfolio Carbon Intensity: {port_carb:.1f} tCO2e / $M Revenue")

--- OPTIMAL STRATEGIC ASSET ALLOCATION (PwC ALM MODEL) ---


,Asset Class,Optimal Weight
0,Adani Green,20.00%
1,NHPC,0.00%
2,Tata Power,0.00%
3,NTPC,0.00%
4,Power Grid,0.00%
5,Coal India,0.00%
6,10Y G-Sec,40.00%
7,30Y Green Bond,40.00%



Portfolio Expected Return: 8.40%
Portfolio Duration: 11.52 Years (Target Liability Duration: 13.50 Years)
Portfolio Carbon Intensity: 2.4 tCO2e / $M Revenue
